In [0]:
display(dbutils.fs.ls("abfss://raw@asgairlines007.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://raw@asgairlines007.dfs.core.windows.net/UseCase - Airlines.xlsx,UseCase - Airlines.xlsx,277887,1789052643000


In [0]:
files = dbutils.fs.ls("abfss://raw@asgairlines007.dfs.core.windows.net/")
display(files)

path,name,size,modificationTime
abfss://raw@asgairlines007.dfs.core.windows.net/UseCase - Airlines.xlsx,UseCase - Airlines.xlsx,277887,1789052643000


In [0]:
print(files[0].path)


abfss://raw@asgairlines007.dfs.core.windows.net/UseCase - Airlines.xlsx


In [0]:
%pip install openpyxl

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
display(dbutils.fs.ls("abfss://processed@asgairlines007.dfs.core.windows.net/"))

path,name,size,modificationTime
abfss://processed@asgairlines007.dfs.core.windows.net/flights.csv,flights.csv,123580,1789103459000


In [0]:
df = spark.read.option("header", "true").option("inferSchema", "true").csv(
    "abfss://processed@asgairlines007.dfs.core.windows.net/flights.csv"
)

display(df)

flight_id,airline,source,destination,departure_time,arrival_time,duration
SJ010,SpiceJet,CCU,MAA,2026-04-20T23:38:41.701Z,2026-04-21T02:32:41.701Z,1899-12-31 02:54:00.0000000
AI155,Air India,BOM,CCU,2026-04-20T23:35:41.703Z,2026-04-21T01:23:41.703Z,1899-12-31 01:48:00.0000000
UK094,Vistara,BOM,CCU,2026-04-20T23:26:41.702Z,2026-04-21T01:11:41.702Z,1899-12-31 01:45:00.0000000
AI245,Air India,BOM,CCU,2026-04-20T23:07:41.704Z,2026-04-21T01:43:41.704Z,1899-12-31 02:36:00.0000000
AI192,Air India,MAA,BOM,2026-04-20T23:05:41.703Z,2026-04-21T04:04:41.703Z,1899-12-31 04:59:00.0000000
SJ158,SpiceJet,DEL,HYD,2026-04-20T23:05:41.703Z,2026-04-21T01:24:41.703Z,1899-12-31 02:19:00.0000000
6F196,IndiGo,CCU,MAA,2026-04-20T23:04:41.703Z,2026-04-21T00:47:41.703Z,1899-12-31 01:43:00.0000000
AI080,Air India,BOM,HYD,2026-04-20T23:03:41.702Z,2026-04-21T00:35:41.702Z,1899-12-31 01:32:00.0000000
6F025,IndiGo,BLR,BOM,2026-04-20T23:02:41.701Z,2026-04-20T23:57:41.701Z,1899-12-31 00:55:00.0000000
6F251,UNKNOWN,DEL,BLR,2026-04-20T22:56:41.703Z,2026-04-20T23:37:41.703Z,1899-12-31 00:41:00.0000000


In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))
print("Column names:", df.columns)

display(df.describe())

Rows: 1020
Columns: 7
Column names: ['flight_id', 'airline', 'source', 'destination', 'departure_time', 'arrival_time', 'duration']


summary,flight_id,airline,source,destination,duration
count,1020,979,1020,1020,1020
mean,null,null,null,null,-0.791666666671517
stddev,null,null,null,null,null
min,6F001,Air India,BLR,BLR,-0.791666666671517
max,UK230,Vistara,MAA,MAA,1899-12-31 05:00:00.0000000


In [0]:
df.printSchema()

root
 |-- flight_id: string (nullable = true)
 |-- airline: string (nullable = true)
 |-- source: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- departure_time: timestamp (nullable = true)
 |-- arrival_time: timestamp (nullable = true)
 |-- duration: string (nullable = true)



In [0]:
from pyspark.sql.functions import col, sum, when

print("Missing values:")
missing = df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
])
display(missing)

print("Duplicate rows:", df.count() - df.dropDuplicates().count())

print("Flight ID examples:")
display(df.select("flight_id").distinct().limit(20))

Missing values:


flight_id,airline,source,destination,departure_time,arrival_time,duration
0,41,0,0,0,0,0


Duplicate rows: 15
Flight ID examples:


flight_id
6F174
SJ001
SJ179
UK176
6F129
6F051
6F047
AI178
6F168
6F049


In [0]:
from pyspark.sql.functions import (
    col, trim, when, hour, minute, concat, lit
)

# 1. Remove exact duplicate records
cleaned_df = df.dropDuplicates()

# 2. Standardize text fields
cleaned_df = (
    cleaned_df
    .withColumn("flight_id", trim(col("flight_id")))
    .withColumn("airline", trim(col("airline")))
    .withColumn("source", trim(col("source")))
    .withColumn("destination", trim(col("destination")))
)

# 3. Replace missing/blank airline with Unknown
cleaned_df = cleaned_df.withColumn(
    "airline",
    when(col("airline").isNull() | (col("airline") == ""), "Unknown")
    .otherwise(col("airline"))
)

# 4. Calculate departure and arrival in minutes
dep_minutes = hour("departure_time") * 60 + minute("departure_time")
arr_minutes = hour("arrival_time") * 60 + minute("arrival_time")

# 5. Calculate flight duration, handling overnight flights
cleaned_df = cleaned_df.withColumn(
    "duration_minutes",
    when(arr_minutes < dep_minutes,
         arr_minutes + 1440 - dep_minutes)
    .otherwise(arr_minutes - dep_minutes)
)

# 6. Create route
cleaned_df = cleaned_df.withColumn(
    "route",
    concat(col("source"), lit(" - "), col("destination"))
)

# 7. Flag zero-duration flights as anomalies
cleaned_df = cleaned_df.withColumn(
    "is_anomaly",
    when(col("duration_minutes") == 0, True).otherwise(False)
)

# 8. Validate flight ID format
cleaned_df = cleaned_df.withColumn(
    "invalid_flight_id",
    ~col("flight_id").rlike("^[A-Za-z0-9]{5}$")
)

display(cleaned_df)

flight_id,airline,source,destination,departure_time,arrival_time,duration,duration_minutes,route,is_anomaly,invalid_flight_id
UK167,Vistara,DEL,MAA,2026-04-20T22:45:41.703Z,2026-04-21T01:46:41.703Z,1899-12-31 03:01:00.0000000,181,DEL - MAA,false,false
SJ023,SpiceJet,HYD,MAA,2026-04-20T17:27:41.701Z,2026-04-20T19:09:41.701Z,1899-12-31 01:42:00.0000000,102,HYD - MAA,false,false
UK207,Vistara,MAA,CCU,2026-04-20T16:21:41.703Z,2026-04-20T17:36:41.703Z,1899-12-31 01:15:00.0000000,75,MAA - CCU,false,false
SJ088,SpiceJet,CCU,MAA,2026-04-20T13:54:41.702Z,2026-04-20T14:26:41.702Z,1899-12-31 00:32:00.0000000,32,CCU - MAA,false,false
UK134,Vistara,HYD,CCU,2026-04-20T13:44:41.702Z,2026-04-20T17:26:41.702Z,1899-12-31 03:42:00.0000000,222,HYD - CCU,false,false
UK212,Vistara,HYD,MAA,2026-04-20T10:46:41.704Z,2026-04-20T11:31:41.704Z,1899-12-31 00:45:00.0000000,45,HYD - MAA,false,false
6F209,IndiGo,MAA,CCU,2026-04-20T08:40:41.703Z,2026-04-20T13:40:41.703Z,1899-12-31 05:00:00.0000000,300,MAA - CCU,false,false
UK193,Vistara,HYD,DEL,2026-04-20T08:39:41.703Z,2026-04-20T09:14:41.703Z,1899-12-31 00:35:00.0000000,35,HYD - DEL,false,false
6F092,IndiGo,MAA,BLR,2026-04-20T08:39:41.702Z,2026-04-20T09:15:41.702Z,1899-12-31 00:36:00.0000000,36,MAA - BLR,false,false
SJ061,SpiceJet,MAA,BLR,2026-04-20T06:38:41.701Z,2026-04-20T09:35:41.701Z,1899-12-31 02:57:00.0000000,177,MAA - BLR,false,false


In [0]:
display(
    cleaned_df.select(
        "flight_id",
        "departure_time",
        "arrival_time",
        "duration_minutes",
        "route",
        "is_anomaly"
    ).limit(20)
)

flight_id,departure_time,arrival_time,duration_minutes,route,is_anomaly
UK167,2026-04-20T22:45:41.703Z,2026-04-21T01:46:41.703Z,181,DEL - MAA,false
SJ023,2026-04-20T17:27:41.701Z,2026-04-20T19:09:41.701Z,102,HYD - MAA,false
UK207,2026-04-20T16:21:41.703Z,2026-04-20T17:36:41.703Z,75,MAA - CCU,false
SJ088,2026-04-20T13:54:41.702Z,2026-04-20T14:26:41.702Z,32,CCU - MAA,false
UK134,2026-04-20T13:44:41.702Z,2026-04-20T17:26:41.702Z,222,HYD - CCU,false
UK212,2026-04-20T10:46:41.704Z,2026-04-20T11:31:41.704Z,45,HYD - MAA,false
6F209,2026-04-20T08:40:41.703Z,2026-04-20T13:40:41.703Z,300,MAA - CCU,false
UK193,2026-04-20T08:39:41.703Z,2026-04-20T09:14:41.703Z,35,HYD - DEL,false
6F092,2026-04-20T08:39:41.702Z,2026-04-20T09:15:41.702Z,36,MAA - BLR,false
SJ061,2026-04-20T06:38:41.701Z,2026-04-20T09:35:41.701Z,177,MAA - BLR,false


In [0]:
output_path = "abfss://processed@asgairlines007.dfs.core.windows.net/cleaned_flights"

cleaned_df.write.mode("overwrite").option("header", "true").csv(output_path)

print("Cleaned dataset saved successfully.")
print("Records:", cleaned_df.count())

Cleaned dataset saved successfully.
Records: 1005


In [0]:
from pyspark.sql.functions import avg, count, desc

print("=== KPI SUMMARY ===")

print("Total Flights:", cleaned_df.count())

print("Average Flight Duration (minutes):")
display(cleaned_df.select(avg("duration_minutes").alias("avg_duration_minutes")))

print("Flights by Airline:")
display(
    cleaned_df.groupBy("airline")
    .count()
    .orderBy(desc("count"))
)

print("Top Routes:")
display(
    cleaned_df.groupBy("route")
    .count()
    .orderBy(desc("count"))
)

print("Anomalies:")
display(
    cleaned_df.groupBy("is_anomaly")
    .count()
)

=== KPI SUMMARY ===
Total Flights: 1005
Average Flight Duration (minutes):


avg_duration_minutes
164.61990049751245


Flights by Airline:


airline,count
IndiGo,249
SpiceJet,236
Air India,233
Vistara,218
Unknown,39
UNKNOWN,30


Top Routes:


route,count
BOM - CCU,90
CCU - DEL,72
MAA - BLR,65
BLR - BOM,60
HYD - MAA,57
DEL - HYD,54
HYD - DEL,42
BOM - DEL,39
CCU - BOM,33
DEL - BLR,29


Anomalies:


is_anomaly,count
false,1005


In [0]:
from pyspark.sql.functions import lower, trim, when, col

cleaned_df = cleaned_df.withColumn(
    "airline",
    when(
        col("airline").isNull() |
        (trim(col("airline")) == "") |
        (lower(trim(col("airline"))) == "unknown"),
        "Unknown"
    ).otherwise(trim(col("airline")))
)

# Save corrected cleaned dataset
cleaned_df.write.mode("overwrite").option("header", "true").csv(
    "abfss://processed@asgairlines007.dfs.core.windows.net/cleaned_flights"
)

print("Final cleaned records:", cleaned_df.count())

display(
    cleaned_df.groupBy("airline")
    .count()
    .orderBy(col("count").desc())
)


Final cleaned records: 1005


airline,count
IndiGo,249
SpiceJet,236
Air India,233
Vistara,218
Unknown,69


In [0]:
final_path = "abfss://processed@asgairlines007.dfs.core.windows.net/final_flights"

cleaned_df.coalesce(1).write.mode("overwrite").option("header", "true").csv(final_path)

print("Final Power BI dataset created.")

Final Power BI dataset created.


In [0]:
display(dbutils.fs.ls(final_path))

path,name,size,modificationTime
abfss://processed@asgairlines007.dfs.core.windows.net/final_flights/_SUCCESS,_SUCCESS,0,1789105542000
abfss://processed@asgairlines007.dfs.core.windows.net/final_flights/_committed_6728478487044878462,_committed_6728478487044878462,113,1789105542000
abfss://processed@asgairlines007.dfs.core.windows.net/final_flights/_started_6728478487044878462,_started_6728478487044878462,0,1789105542000
abfss://processed@asgairlines007.dfs.core.windows.net/final_flights/part-00000-tid-6728478487044878462-abf6844c-0bf3-4949-96d8-6c52e78073ca-174-1-c000.csv,part-00000-tid-6728478487044878462-abf6844c-0bf3-4949-96d8-6c52e78073ca-174-1-c000.csv,126945,1789105542000


In [0]:
part_file = [f.path for f in dbutils.fs.ls(final_path) if f.name.startswith("part-")][0]

dbutils.fs.cp(
    part_file,
    "abfss://processed@asgairlines007.dfs.core.windows.net/final_flights.csv"
)

print("CSV ready for Power BI")

CSV ready for Power BI
